In [1]:
%load_ext autoreload

In [2]:
%autoreload 2

In [3]:
import mynnlib
from mynnlib import *

dataset_dir = "insect-dataset/moth"

early_regex = r"^.*-(early)$"
unidentified_regex = r"^.*-(spp|genera|genera-spp)$"
early_or_unidentified_regex = r"^.*-(early|spp|genera|genera-spp)$"

# Count

In [80]:
classes = { class_dir: len([ img for img in os.listdir(f"{dataset_dir}/data/{class_dir}") ]) for class_dir in os.listdir(f"{dataset_dir}/data") }
early_classes = { class_name: count for class_name, count in classes.items() if re.match(early_regex, class_name) }
unidentified_classes = { class_name: count for class_name, count in classes.items() if re.match(unidentified_regex, class_name) }
print(f"Total Class count : {len(classes):6} ( Unidentified: {len(unidentified_classes):6} / Early-stage: {len(early_classes):6} / Identified-adult: {len(classes) - len(unidentified_classes) - len(early_classes):6} )")
print(f"Total  Data count : {sum(classes.values()):6} ( Unidentified: {sum(unidentified_classes.values()):6} / Early-stage: {sum(early_classes.values()):6} / Identified-adult: {sum(classes.values()) - sum(unidentified_classes.values()) - sum(early_classes.values()):6} )")

Total Class count :   3364 ( Unidentified:    411 / Early-stage:    304 / Identified-adult:   2649 )
Total  Data count :  44652 ( Unidentified:  11156 / Early-stage:   3569 / Identified-adult:  29927 )


In [81]:
img2_class = []
img5_class = []
for class_dir in os.listdir(f"{dataset_dir}/data"):
    if not re.match(early_or_unidentified_regex, class_dir):
        img_cnt = sum([1 for file in os.listdir(f"{dataset_dir}/data/{class_dir}")])
        img2_class += [class_dir] if img_cnt <= 2 else []
        img5_class += [class_dir] if img_cnt <= 5 else []
print(f"{len(img2_class):6} classes with <=2 images")
print(f"{len(img5_class):6} classes with <=5 images")

   548 classes with <=2 images
  1276 classes with <=5 images


In [91]:
generas = set()
for class_name in classes:
    generas.add(class_name.split('-')[0])
print(f"Genera count: {len(generas)}")

Genera count: 1462


# Add more data

In [82]:
def copy_data_from(sources):
    class_cnt = 0
    img_cnt = 0
    for more_data_dir in sources:
        for class_dir in os.listdir(f"{dataset_dir}/data"):
            if os.path.exists(f"{more_data_dir}/{class_dir}"):
                # print(f"Copying data for {class_dir}...")
                class_cnt += 1
                for file in os.listdir(f"{more_data_dir}/{class_dir}"):
                    shutil.copy2(f"{more_data_dir}/{class_dir}/{file}", f"{dataset_dir}/data/{class_dir}/{file}")
                    img_cnt += 1
    print(f"{img_cnt} images added into {class_cnt} classes")

In [83]:
copy_data_from(["insect-dataset/lepidoptera.indiabiodiversity.org.2025.02.10"])

1958 images added into 587 classes


In [84]:
copy_data_from(["insect-dataset/wikipedia.org"])

1260 images added into 1064 classes


In [85]:
copy_data_from(["insect-dataset/insecta.pro"])

1201 images added into 395 classes


In [111]:
copy_data_from(["insect-dataset/inaturalist.org"])

22330 images added into 800 classes


In [117]:
# remove early classes
for class_dir in os.listdir(f"{dataset_dir}/data"):
    if class_dir.endswith("-early"):
        shutil.rmtree(f"{dataset_dir}/data/{class_dir}")

In [118]:
classes = { class_dir: len([ img for img in os.listdir(f"{dataset_dir}/data/{class_dir}") ]) for class_dir in os.listdir(f"{dataset_dir}/data") }
early_classes = { class_name: count for class_name, count in classes.items() if re.match(early_regex, class_name) }
unidentified_classes = { class_name: count for class_name, count in classes.items() if re.match(unidentified_regex, class_name) }
print(f"Total Class count : {len(classes):6} ( Unidentified: {len(unidentified_classes):6} / Early-stage: {len(early_classes):6} / Identified-adult: {len(classes) - len(unidentified_classes) - len(early_classes):6} )")
print(f"Total  Data count : {sum(classes.values()):6} ( Unidentified: {sum(unidentified_classes.values()):6} / Early-stage: {sum(early_classes.values()):6} / Identified-adult: {sum(classes.values()) - sum(unidentified_classes.values()) - sum(early_classes.values()):6} )")

Total Class count :   3060 ( Unidentified:    411 / Early-stage:      0 / Identified-adult:   2649 )
Total  Data count :  67800 ( Unidentified:  11156 / Early-stage:      0 / Identified-adult:  56644 )


In [119]:
img2_class = []
img5_class = []
for class_dir in os.listdir(f"{dataset_dir}/data"):
    if not re.match(early_or_unidentified_regex, class_dir):
        img_cnt = sum([1 for file in os.listdir(f"{dataset_dir}/data/{class_dir}")])
        img2_class += [class_dir] if img_cnt <= 2 else []
        img5_class += [class_dir] if img_cnt <= 5 else []
print(f"{len(img2_class):6} classes with <=2 images")
print(f"{len(img5_class):6} classes with <=5 images")

   131 classes with <=2 images
   638 classes with <=5 images


# Train

### Model A

In [5]:
training_params = [
    { "idx": "01", "robustness": 0.2, "break_at_val_acc_diff": 0.05},
    { "idx": "02", "robustness": 0.5, "break_at_val_acc_diff": 0.02},
    { "idx": "03", "robustness": 1.0, "break_at_val_acc_diff": 0.01},
    { "idx": "04", "robustness": 2.0, "break_at_val_acc_diff": 0.005}
]
for param in training_params:
    if param["idx"] == "01":
        model_data = init_model_for_training(f'{dataset_dir}/data', f'{dataset_dir}/val', 
                                             batch_size=32, arch="resnet101", image_size=224, robustness=0.2,
                                             lr=1e-4, weight_decay=1e-4, silent=True)
    else:
        model_data = prepare_for_retraining(model_data, f'{dataset_dir}/data', f'{dataset_dir}/val', 
                                            batch_size=32, image_size=224, robustness=param["robustness"], silent=True)
    train(model_data, 5, f"{dataset_dir}/checkpoint.moth.ta.ep{param["idx"]}###.pth", 
          break_at_val_acc_diff=param["break_at_val_acc_diff"])

Epoch    1 /    5  | Train Loss: 3.5788 Acc: 0.4362  | Val Loss: 1.6324 Acc: 0.5844  | Elapsed time: 0:13:59.920457
Epoch    2 /    5  | Train Loss: 0.7470 Acc: 0.8189  | Val Loss: 1.2188 Acc: 0.7143  | Elapsed time: 0:28:33.046725
Epoch    3 /    5  | Train Loss: 0.3673 Acc: 0.9033  | Val Loss: 1.3251 Acc: 0.7013  | Elapsed time: 0:43:07.716677
Epoch    1 /    5  | Train Loss: 1.3027 Acc: 0.7161  | Val Loss: 1.1361 Acc: 0.7143  | Elapsed time: 0:15:17.443942
Epoch    2 /    5  | Train Loss: 1.0163 Acc: 0.7729  | Val Loss: 1.0305 Acc: 0.7273  | Elapsed time: 0:30:44.161012
Epoch    1 /    5  | Train Loss: 0.9890 Acc: 0.7799  | Val Loss: 1.1097 Acc: 0.7403  | Elapsed time: 0:15:18.878553
Epoch    2 /    5  | Train Loss: 0.9098 Acc: 0.7968  | Val Loss: 1.0986 Acc: 0.7338  | Elapsed time: 0:30:49.874986
Epoch    1 /    5  | Train Loss: 0.8090 Acc: 0.8220  | Val Loss: 0.9465 Acc: 0.7597  | Elapsed time: 0:15:31.187134
Epoch    2 /    5  | Train Loss: 0.7119 Acc: 0.8429  | Val Loss: 0.9149 

In [6]:
train(model_data, 5, f"{dataset_dir}/checkpoint.moth.ta.ep05###.pth", break_at_val_acc_diff=0.005)

Epoch    1 /    5  | Train Loss: 0.6779 Acc: 0.8503  | Val Loss: 0.9450 Acc: 0.7597  | Elapsed time: 0:15:18.200399
Epoch    2 /    5  | Train Loss: 0.6501 Acc: 0.8551  | Val Loss: 0.9046 Acc: 0.7792  | Elapsed time: 0:30:41.719849
Epoch    3 /    5  | Train Loss: 0.6312 Acc: 0.8615  | Val Loss: 0.9144 Acc: 0.7662  | Elapsed time: 0:45:59.829237


### Model B

In [ ]:
training_params = [
    { "idx": 1, "robustness": 0.2, "break_at_val_acc_diff": 0.05},
    { "idx": 2, "robustness": 0.5, "break_at_val_acc_diff": 0.02},
    { "idx": 3, "robustness": 1.0, "break_at_val_acc_diff": 0.01},
    { "idx": 4, "robustness": 2.0, "break_at_val_acc_diff": -0.000001}
]
for param in training_params:
    print(f"Phase {param["idx"]}:")
    if param["idx"] == 1:
        model_data = init_model_for_training(f'{dataset_dir}/data', f'{dataset_dir}/val', 
                                             batch_size=32, arch="resnet152", image_size=224, robustness=param["robustness"],
                                             lr=1e-4, weight_decay=1e-4, silent=True)
    else:
        model_data = prepare_for_retraining(model_data, f'{dataset_dir}/data', f'{dataset_dir}/val', 
                                            batch_size=32, image_size=224, robustness=param["robustness"], silent=True)
    train(model_data, 5, f"{dataset_dir}/checkpoint.moth.tb.ep{param["idx"]:02}###.pth", 
          break_at_val_acc_diff=param["break_at_val_acc_diff"])

Phase 1:
Epoch    1 /    5 